# 18차시 실습 — 내 서비스를 '운영'하기 (대시보드·알림·드리프트)

> **day1~3 연속 실습.** 17차시에서 LLM 호출에 **계측**(`@observe`·토큰/비용·`score`)을 붙였습니다.
> 오늘은 그 신호를 **운영**합니다 — 지표를 **집계(미니 대시보드)** 하고, **알림 규칙**으로 이상을 잡고,
> **드리프트·카나리아**로 조용한 표류를 감지한 뒤, **내 팀 MVP에 적용**합니다.

## 🧪 사용법
- 이 노트북은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다. 위에서부터 **순서대로**(`Shift+Enter`).
- **계측 자체(Langfuse `@observe`)는 17차시 노트북에서 했습니다.** 오늘은 **로컬 집계 로거**로 대시보드·알림을 직접 만들어 원리를 익히고(1~4단계), 같은 화면을 **Langfuse 대시보드**에서 읽는 법을 봅니다(5단계).
- **Langfuse 없이 전 단계가 실행됩니다**(키 불필요).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

day1: **맛집 추천 도우미** MVP를 만들었습니다(7차시에서 에이전트로 발전).
day3: 같은 서비스를 **운영**한다고 가정 — 사용자 요청을 처리하는 **AI 호출이 빠른지·싼지·실패하는지·똑똑한지**를 숫자로 봅니다.

> 구조는 동일하니, 아래 로거를 **당신 팀 MVP의 LLM 호출**에 그대로 감싸면 적용됩니다(8단계).

## 1단계 — 모니터링용 집계 로거 (키 불필요)

17차시에서 `@observe`로 trace를 남겼다면, 오늘은 그 신호를 **집계·알림**하기 위해
요청마다 **지연·토큰·성공/실패·품질점수**를 평평한 레코드로 모으는 작은 로거를 만듭니다.
(실전에선 이 집계를 **Langfuse 대시보드가 대신** 해줍니다 — 5단계에서 그 화면을 읽습니다.)

⌨️ **터미널 Codex:** `codex "요청마다 지연·토큰·성공여부·품질점수를 레코드로 모으는 모니터링 로거를 만들어줘"`

In [2]:
import time

OBSERVATIONS = []   # 로컬 관찰성 저장소 (실제로는 Langfuse/DB로 전송할 자리)

def quality_score(question, answer):
    """초간단 휴리스틱 품질점수(0~1). 실제로는 LLM 심판/규칙/사용자 피드백을 사용."""
    if not answer or "오류" in answer:
        return 0.0
    score = 0.5
    if len(answer) > 20:                                   # 너무 짧지 않음
        score += 0.25
    if answer.strip().endswith(("다", "다.", ".", "요")):  # 문장으로 마무리
        score += 0.25
    return round(min(score, 1.0), 2)

# 내 서비스(맛집 추천 도우미)의 역할 — 팀 MVP에 맞게 바꾸세요
SERVICE_SYSTEM = "너는 맛집 추천 도우미다. 사용자 요청에 간결하고 정확하게 한국어로 답하라."

def observed_llm_call(question, system=SERVICE_SYSTEM):
    """내 서비스의 LLM 호출을 계측: 지연·토큰·성공여부·품질점수를 기록하고 record를 반환."""
    t0 = time.time()
    record = {"question": question, "ok": False, "latency_ms": 0,
              "prompt_tokens": 0, "completion_tokens": 0, "answer": "", "score": 0.0}
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": question}],
        )
        answer = resp.choices[0].message.content
        u = resp.usage
        record.update(ok=True, answer=answer,
                      prompt_tokens=u.prompt_tokens,
                      completion_tokens=u.completion_tokens,
                      score=quality_score(question, answer))
    except Exception as e:
        record["answer"] = f"오류: {e}"
    record["latency_ms"] = round((time.time() - t0) * 1000)
    OBSERVATIONS.append(record)          # ← 관찰 기록 저장
    return record

print("로거 준비 완료 · 기록 항목: latency_ms / prompt·completion_tokens / ok / score")

로거 준비 완료 · 기록 항목: latency_ms / prompt·completion_tokens / ok / score


## 2단계 — 계측하며 내 서비스 호출하기

실제 사용자가 보낼 법한 요청들을 로거로 흘려보냅니다. 각 호출이 끝날 때마다 `OBSERVATIONS`에 기록이 쌓입니다.

In [3]:
# 내 서비스(맛집 추천 도우미)에 들어올 법한 사용자 요청들
questions = [
    "강남에서 2만원 이하 매운 점심 추천해줘.",
    "홍대 근처 데이트하기 좋은 분위기 식당 알려줘.",
    "혼밥하기 좋은 국밥집 한 곳만 추천해줘.",
]
for q in questions:
    r = observed_llm_call(q)
    print(f"[{'OK ' if r['ok'] else 'FAIL'}] {r['latency_ms']:>5}ms · score={r['score']} · {r['question'][:24]}…")

[OK ]  6819ms · score=0.75 · 강남에서 2만원 이하 매운 점심 추천해줘.…
[OK ]  3505ms · score=0.75 · 홍대 근처 데이트하기 좋은 분위기 식당 알려…
[OK ]  1688ms · score=1.0 · 혼밥하기 좋은 국밥집 한 곳만 추천해줘.…


## 3단계 — 미니 대시보드 (지표 집계 + 비용 가시화)

쌓인 기록을 표로 보고, **성공률·평균지연·평균품질·총비용**을 한 줄로 요약합니다.
토큰을 **비용($)** 으로 환산하면 "어느 요청이 비싼가"가 바로 보입니다.

⌨️ **터미널 Codex:** `codex "관찰 기록 리스트를 받아 성공률·평균지연·평균점수·총비용을 출력하는 대시보드 함수를 만들어줘"`

In [4]:
# gpt-4o-mini 예시 단가 (1M 토큰당 USD) — 실제 단가는 공급사 가격 페이지에서 확인
PRICE_IN, PRICE_OUT = 0.15, 0.60

def cost_usd(r):
    return (r["prompt_tokens"] * PRICE_IN + r["completion_tokens"] * PRICE_OUT) / 1_000_000

def dashboard(rows):
    print(f"{'요청':<26}{'상태':<6}{'지연(ms)':>9}{'토큰':>7}{'점수':>6}{'비용$':>11}")
    print("-" * 66)
    for r in rows:
        tok = r["prompt_tokens"] + r["completion_tokens"]
        state = "OK" if r["ok"] else "FAIL"
        print(f"{r['question'][:24]:<26}{state:<6}{r['latency_ms']:>9}{tok:>7}{r['score']:>6}{cost_usd(r):>11.6f}")
    print("-" * 66)
    n = len(rows) or 1
    ok = sum(r["ok"] for r in rows)
    print(f"요청 {len(rows)}건 · 성공률 {ok / n * 100:.0f}% · "
          f"평균지연 {sum(r['latency_ms'] for r in rows) // n}ms · "
          f"평균점수 {sum(r['score'] for r in rows) / n:.2f} · "
          f"총비용 ${sum(cost_usd(r) for r in rows):.6f}")

dashboard(OBSERVATIONS)

요청                        상태       지연(ms)     토큰    점수        비용$
------------------------------------------------------------------
강남에서 2만원 이하 매운 점심 추천해줘.   OK         6819    232  0.75   0.000116
홍대 근처 데이트하기 좋은 분위기 식당 알려  OK         3505    215  0.75   0.000106
혼밥하기 좋은 국밥집 한 곳만 추천해줘.    OK         1688    109   1.0   0.000042
------------------------------------------------------------------
요청 3건 · 성공률 100% · 평균지연 4004ms · 평균점수 0.83 · 총비용 $0.000264


## 4단계 — 알림 규칙 (이상 자동 플래그)

관찰의 목적은 **수집**이 아니라 **행동**입니다. 임계값을 정해 두면 이상 요청을 자동으로 표시할 수 있습니다.
규칙 예: **실패** / **지연 > 임계값** / **품질점수 < 하한**.

⌨️ **터미널 Codex:** `codex "관찰 기록에서 실패·고지연·저품질을 플래그하는 알림 규칙 함수를 만들어줘"`

In [5]:
LATENCY_LIMIT_MS = 4000   # 지연 임계값 (내 서비스 기준으로 조정)
SCORE_FLOOR = 0.5         # 품질 하한

def check_alerts(rows):
    alerts = []
    for r in rows:
        if not r["ok"]:
            alerts.append((r["question"][:24], "실패(에러)"))
        elif r["latency_ms"] > LATENCY_LIMIT_MS:
            alerts.append((r["question"][:24], f"고지연 {r['latency_ms']}ms > {LATENCY_LIMIT_MS}"))
        elif r["score"] < SCORE_FLOOR:
            alerts.append((r["question"][:24], f"저품질 score {r['score']} < {SCORE_FLOOR}"))
    if not alerts:
        print("🟢 알림 없음 — 모든 지표 정상")
    for q, why in alerts:
        print(f"🔴 ALERT · {q}… → {why}")

check_alerts(OBSERVATIONS)

🔴 ALERT · 강남에서 2만원 이하 매운 점심 추천해줘.… → 고지연 6819ms > 4000


## 5단계 — 같은 지표를 Langfuse 대시보드에서 '운영'하기

1~4단계에서 **직접** 집계·알림을 만들어 원리를 익혔습니다. 실전에선 이걸 손으로 안 만듭니다 —
**계측은 이미 17차시에서 붙였습니다**(`@observe` · `langfuse.openai` · `score_current_trace`).
그 신호가 Langfuse에 쌓이면, 오늘 만든 대시보드/알림을 **Langfuse UI에서 그대로** 봅니다.

**Langfuse 대시보드에서 읽는 법 (운영)**
- **Traces**: 느린/실패한 요청 하나를 골라 단계까지 드릴다운 → 원인 파악
- **Latency**: P50/P95/P99 추세 — 평균이 아니라 꼬리(P95)를 본다
- **Cost / Tokens**: 모델·세션별 비용 — 어떤 기능이 비싼가
- **Scores**: 우리가 올린 품질점수 추세 — 회귀를 사용자보다 먼저

**알림(Alert) 걸기**: Langfuse UI에서 `지연 > 임계` · `score < 하한` 조건을 걸면,
4단계의 `check_alerts`를 **사람이 안 지켜봐도** 자동으로 Slack·이메일로 보냅니다.

> 계측 코드(`@observe`·드롭인·`score`)는 **17차시 노트북**에서 다뤘으므로 여기서 반복하지 않습니다.
> 17차시 노트북을 실행해 뒀다면 이미 trace가 쌓여 있습니다 — `localhost:3000`에서 위 네 탭을 열어 확인하세요.

## 6단계 — 드리프트 감지 맛보기: 분포가 조용히 변한다

강의의 KS/KL/PSI — 원리는 '두 구간의 분포 비교'입니다. 질문 길이 분포로 입력 드리프트를 흉내 냅니다.

In [6]:
import statistics
period_a = ["강남 매운 점심", "홍대 순한 국밥", "2만원 이하 추천"]          # 평소 입력
period_b = ["우리 회사 3분기 매출 보고서를 표로 정리하고 리스크를 분석해줘",   # 갑자기 달라진 입력
            "계약서 조항 7.2의 법적 쟁점을 설명해줘"]
avg_a, avg_b = statistics.mean(map(len, period_a)), statistics.mean(map(len, period_b))
shift = abs(avg_b - avg_a) / max(avg_a, 1)
print(f"구간 A 평균 길이 {avg_a:.0f}자 → 구간 B {avg_b:.0f}자 (변화율 {shift:.0%})")
print("⚠ 입력 드리프트 의심 — 사용자가 다른 용도로 쓰기 시작했다. 평가셋·프롬프트 재점검!" if shift > 0.5 else "✅ 분포 안정")
# 실전: 길이 대신 임베딩 분포에 KS/KL/PSI를 적용한다 (임계: KS>0.1, PSI>0.25)

구간 A 평균 길이 8자 → 구간 B 29자 (변화율 248%)
⚠ 입력 드리프트 의심 — 사용자가 다른 용도로 쓰기 시작했다. 평가셋·프롬프트 재점검!


## 7단계 — 카나리아: 정답을 아는 질문을 주기적으로 발사

강의의 합성 모니터링 — 사용자보다 먼저 회귀를 발견합니다.

In [7]:
CANARY_Q = "강남에서 매운 음식 추천해줘"        # 정답 경향을 아는 질문
CANARY_FLOOR = 0.5
scores = []
for i in range(3):
    rec = observed_llm_call(CANARY_Q)          # 로거가 지연·토큰·score까지 기록
    scores.append(rec["score"])
    print(f"  카나리아 {i+1}: score={rec['score']} · {rec['latency_ms']}ms")
import statistics as _st
avg = _st.mean(scores)
print(f"평균 {avg:.2f} → " + ("🚨 임계 미달 — 알림! 실패 케이스를 골든셋에 추가하라" if avg < CANARY_FLOOR else "✅ 정상 — 다음 주기에 다시"))

  카나리아 1: score=0.75 · 4012ms
  카나리아 2: score=0.75 · 2397ms
  카나리아 3: score=0.75 · 2731ms
평균 0.75 → ✅ 정상 — 다음 주기에 다시


## 8단계 — ⭐ 내 팀 MVP에 적용 (실패 케이스도 잡아보기)

위 구조(로거 → 대시보드 → 알림)는 **그대로** 당신 팀 MVP의 AI 호출에 옮길 수 있습니다.
운영에서 가장 중요한 건 **실패와 저품질을 놓치지 않는 것** — 일부러 **실패하는 호출**을 넣어 알림에 잡히는지 확인하세요.

**내 MVP에 적용하는 법**
1. MVP의 **AI 호출 지점**을 `observed_llm_call(...)`로 감싼다.
2. 우리 도메인에 맞게 **`quality_score`** 와 **`SERVICE_SYSTEM`** 을 수정한다.
3. 알림 임계값(`LATENCY_LIMIT_MS`, `SCORE_FLOOR`)을 우리 서비스 기준으로 조정한다.

⌨️ **터미널 Codex:** `codex "우리 MVP의 LLM 호출에 관찰성 로거를 붙이고 대시보드·알림 규칙을 추가해줘"`

In [8]:
# 일부러 실패시키기 — 존재하지 않는 모델명으로 호출 (try/except로 FAIL 기록됨)
_real_model = MODEL
MODEL = "gpt-this-model-does-not-exist"
observed_llm_call("이 호출은 실패할 거예요. 알림에 잡히나요?")
MODEL = _real_model   # 원래 모델로 복구

# 다시 집계 + 알림 — 실패 행이 FAIL/🔴 로 잡히는지 확인
print("=== 대시보드 ===")
dashboard(OBSERVATIONS)
print("\n=== 알림 ===")
check_alerts(OBSERVATIONS)

# 팀별 작성: 우리 MVP의 AI 호출 지점을 적어보세요 → 위 observed_llm_call로 감싸면 끝
MY_AI_CALLS_TODO = [
  # "예: 사용자 글을 요약하는 호출",
  # "예: 추천 사유를 생성하는 호출",
]
print("\n우리 MVP에서 관찰할 AI 호출:", MY_AI_CALLS_TODO or "⬜ 채워주세요")

=== 대시보드 ===
요청                        상태       지연(ms)     토큰    점수        비용$
------------------------------------------------------------------
강남에서 2만원 이하 매운 점심 추천해줘.   OK         6819    232  0.75   0.000116
홍대 근처 데이트하기 좋은 분위기 식당 알려  OK         3505    215  0.75   0.000106
혼밥하기 좋은 국밥집 한 곳만 추천해줘.    OK         1688    109   1.0   0.000042
강남에서 매운 음식 추천해줘           OK         4012    235  0.75   0.000121
강남에서 매운 음식 추천해줘           OK         2397    164  0.75   0.000078
강남에서 매운 음식 추천해줘           OK         2731    170  0.75   0.000082
이 호출은 실패할 거예요. 알림에 잡히나요?  FAIL        213      0   0.0   0.000000
------------------------------------------------------------------
요청 7건 · 성공률 86% · 평균지연 3052ms · 평균점수 0.68 · 총비용 $0.000545

=== 알림 ===
🔴 ALERT · 강남에서 2만원 이하 매운 점심 추천해줘.… → 고지연 6819ms > 4000
🔴 ALERT · 강남에서 매운 음식 추천해줘… → 고지연 4012ms > 4000
🔴 ALERT · 이 호출은 실패할 거예요. 알림에 잡히나요?… → 실패(에러)

우리 MVP에서 관찰할 AI 호출: ⬜ 채워주세요


## 정리

- 17차시에서 **계측**(trace·토큰·score)을 붙였다면, 18차시는 그 신호를 **운영**한다 — 집계·알림·드리프트.
- **미니 대시보드**(성공률·P95·비용·점수)와 **알림 규칙**(실패·고지연·저품질)으로 이상을 자동으로 잡는다.
- **드리프트·카나리아**로 조용한 표류를 사용자보다 먼저 발견한다.
- 핵심은 수집이 아니라 **행동** — 지표마다 알림·롤백·모델 티어 조정. 다음 차시(19): 이 신호로 **지속 개선**.